# AI Speak — inference

Generates ARKit blendshape animation (52 coefficients, 60 FPS) for every `.wav` / `.mp3` in `INPUT_DIR`.

For each audio file it writes
- `<name>.csv`, one row per frame and 52 columns (values 0–1)
- `meta.json`, with inference time, real-time factor (RTF), lookahead and output FPS for every file

Needs `best_model_v1.pth` (from `train.ipynb` or this repo).

## 1. Setup

In [ ]:
!pip install transformers librosa soundfile -q

import os
import time
import json
import numpy as np
import librosa
import torch
import torch.nn as nn
from transformers import HubertModel, Wav2Vec2FeatureExtractor
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 2. Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 👉 Change these to your own folders
BASE       = '/content/drive/MyDrive/Colab Notebooks/AISpeak_Libraries'
MODEL_PATH = os.path.join(BASE, 'best_model_v1.pth')
INPUT_DIR  = os.path.join(BASE, 'test')              # audio files to animate
OUTPUT_DIR = os.path.join(BASE, 'submission_final')  # CSVs + meta.json go here

## 3. Load models

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
hubert.eval()
hubert.to(DEVICE)


class BlendshapeModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, output_dim=52):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )
        self.output_layers = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        center = lstm_out[:, lstm_out.shape[1] // 2, :]
        return self.output_layers(center)


model = BlendshapeModel()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Models loaded.")

## 4. Inference and post-processing

Post-processing steps
1. Suppress the inhale at the start (first ~0.7 s, low energy)
2. Amplify mouth blendshapes
3. Close the mouth when audio energy drops (silence)
4. Smooth eyes and brows (σ = 8)
5. Add procedural blinks every 3–5 s
6. Clip to [0, 1] and smooth the lower face (σ = 1.5)

In [ ]:
CONTEXT = 5        # 5 frames before + 5 after → 83.3 ms lookahead at 60 FPS
OUTPUT_FPS = 60


def align_features(hubert_features, num_frames):
    h_times = np.linspace(0, 1, len(hubert_features))
    t_times = np.linspace(0, 1, num_frames)
    return interp1d(h_times, hubert_features, axis=0)(t_times).astype(np.float32)


def inference(audio_path):
    start_time = time.time()

    audio, sr = librosa.load(audio_path, sr=16000)
    audio_duration = len(audio) / sr

    with torch.no_grad():
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
        feats = hubert(inputs.input_values.to(DEVICE)).last_hidden_state
        feats = feats.squeeze(0).cpu().numpy()

    num_frames = int(audio_duration * OUTPUT_FPS)
    aligned = align_features(feats, num_frames)

    pad = np.zeros((CONTEXT, aligned.shape[1]), dtype=np.float32)
    padded = np.concatenate([pad, aligned, pad], axis=0)

    preds = []
    with torch.no_grad():
        windows = np.array([padded[i:i + 2*CONTEXT + 1] for i in range(num_frames)])
        for start in range(0, len(windows), 256):
            batch = torch.tensor(windows[start:start+256], dtype=torch.float32).to(DEVICE)
            preds.append(model(batch).cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    inf_time = time.time() - start_time
    return preds, inf_time, audio_duration


def postprocess(predictions, audio_path, sigma=1.5):

    result = predictions.copy()

    # Frame energy, used to detect silence and the inhale
    audio, sr = librosa.load(audio_path, sr=16000)
    hop = sr // 60
    num_frames = len(result)

    energy = np.array([
        np.sqrt(np.mean(audio[i*hop:(i+1)*hop] ** 2))
        for i in range(min(num_frames, len(audio) // hop))
    ])
    if len(energy) < num_frames:
        energy = np.pad(energy, (0, num_frames - len(energy)))
    if energy.max() > 0:
        energy = energy / energy.max()

    smooth_energy = gaussian_filter1d(energy, sigma=3)

    # 1. Suppress the inhale at the start
    for i in range(min(42, num_frames)):
        if energy[i] < 0.15:
            result[i, 19:52] *= 0.05

    # 2. Amplify mouth blendshapes
    mouth_amplify = {
        24: 1.7, 28: 1.2, 29: 1.0, 36: 1.5,
        43: 1.8, 44: 1.8, 45: 1.8, 46: 1.8,
        37: 1.3, 38: 1.3,
    }
    for idx, scale in mouth_amplify.items():
        result[:, idx] = result[:, idx] * scale

    # 3. Close the mouth when energy drops
    mouth_open_indices = [24, 28, 29, 43, 44, 45, 46]
    for i in range(1, num_frames):
        if smooth_energy[i] < 0.08:
            close_factor = smooth_energy[i] / 0.08
            for idx in mouth_open_indices:
                result[i, idx] *= close_factor

    # 4. Smooth eyes and brows
    for idx in range(0, 19):
        result[:, idx] = gaussian_filter1d(
            result[:, idx], sigma=8
        )

    # 5. Procedural blinks (13 = eyeBlinkLeft, 14 = eyeBlinkRight)
    fps = 60
    np.random.seed(42)
    frame = np.random.randint(fps, 3 * fps)
    while frame < num_frames - fps:
        for i in range(10):
            if frame + i < num_frames:
                t = i / 10
                blink_val = np.sin(t * np.pi) * 0.8
                result[frame + i, 13] = max(
                    result[frame + i, 13], blink_val
                )
                result[frame + i, 14] = max(
                    result[frame + i, 14], blink_val
                )
        frame += np.random.randint(3 * fps, 5 * fps)

    result = np.clip(result, 0, 1)
    for idx in range(19, 52):
        result[:, idx] = gaussian_filter1d(
            result[:, idx], sigma=sigma
        )
    return result


## 5. Generate CSVs and meta.json

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional per-file correction. Jovana5 had an exaggerated jaw, so jawOpen (index 24) is scaled down.
# Leave empty ({}) for new audio.
JAWOPEN_SCALE = {'Jovana5': 0.6}

audio_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(('.wav', '.mp3')))

print(f"Files: {len(audio_files)}\n")
print(f"{'File':15s} | {'Duration':>8s} | {'Inference':>10s} | {'RTF':>8s} | {'Frames':>8s}")
print("-" * 65)

all_meta = {}
for audio_file in audio_files:
    audio_path = os.path.join(INPUT_DIR, audio_file)
    name = os.path.splitext(audio_file)[0]

    preds, inf_time, dur = inference(audio_path)
    preds = postprocess(preds, audio_path, sigma=1.5)
    if name in JAWOPEN_SCALE:
        preds[:, 24] *= JAWOPEN_SCALE[name]

    np.savetxt(os.path.join(OUTPUT_DIR, f"{name}.csv"), preds, delimiter=',', fmt='%.3f')

    all_meta[name] = {
        "inference_time_sec": round(inf_time, 4),
        "rtf": round(inf_time / dur, 4),
        "lookahead_ms": 83.3,
        "fps_out": OUTPUT_FPS,
    }
    print(f"{name:15s} | {dur:7.2f}s | {inf_time:9.3f}s | {inf_time/dur:7.4f} | {preds.shape[0]:>8d}")

with open(os.path.join(OUTPUT_DIR, "meta.json"), 'w') as f:
    json.dump(all_meta, f, indent=2)

rtf = [m['rtf'] for m in all_meta.values()]
print(f"\nMean RTF: {np.mean(rtf):.4f} | max RTF: {max(rtf):.4f}")
print(f"Saved to {OUTPUT_DIR}")